# Task decomposition end to end

This notebook is a reusable, stage-by-stage evaluation surface for the canonical task-decomposition APIs. Provider calls are made only by an explicit button after the opt-in checkbox is selected.

After installing this update, restart the kernel and clear all outputs once, then run the notebook from the top. Result panels replace their HTML contents directly; repeated clicks run new experiments. Rerunning the controls cell retires its previous widgets and callbacks.


## 1. Setup and provider configuration

Imports and provider construction are centralized here. Credentials come from the existing environment and are never displayed.

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
from decimal import Decimal
import hashlib, json, os
from html import escape
from typing import Any

try:
    import ipywidgets as widgets
    import pandas as pd
    from IPython.display import Markdown, clear_output, display
except ImportError as exc:
    raise ImportError("This notebook requires pandas, ipywidgets, and IPython. Install the project notebook dependencies.") from exc

from task_decomposition import (
    SupportWorkMode, TaskDecompositionRequest, TaskReference,
    TransformationDecompositionRequest, decompose,
    decompose_task, decompose_transformation,
)

PROVIDER_ENV = {"OpenAI": "OPENAI_API_KEY", "Gemini": "GEMINI_API_KEY", "DeepSeek": "DEEPSEEK_API_KEY"}
state = {"task_result": None, "task_fingerprint": None, "transformation_result": None, "transformation_fingerprint": None, "runs": []}

def _provider(label, allow_live):
    """Build the selected provider only after explicit live-call opt-in."""
    if not allow_live:
        raise RuntimeError("Live provider calls are disabled. Select the opt-in checkbox before running.")
    env_name = PROVIDER_ENV[label]
    if not os.getenv(env_name):
        raise RuntimeError(f"{env_name} is not set; configure credentials in the environment.")
    try:
        if label == "OpenAI":
            from task_decomposition.providers.openai import OpenAIDecompositionProvider
            return OpenAIDecompositionProvider()
        if label == "Gemini":
            from task_decomposition.providers.gemini import GeminiDecompositionProvider
            return GeminiDecompositionProvider()
        from task_decomposition.providers.deepseek import DeepSeekDecompositionProvider
        return DeepSeekDecompositionProvider()
    except ImportError as exc:
        raise ImportError(f"Optional dependency for {label} is missing. Install its project extra.") from exc

def _fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

def _set_panel(panel, *items):
    """Replace one panel value; never publish callback output via IOPub capture."""
    panel.value = "".join(
        item.to_html(index=False, escape=True)
        if isinstance(item, pd.DataFrame) else str(item)
        for item in items
    )

def _error(panel, exc):
    _set_panel(panel, f"<p><strong>Run stopped:</strong> {escape(type(exc).__name__)} — {escape(str(exc))}</p>")


## 2. Example cases

Cases are illustrative assumptions, not validated estimates. Add a case by adding the same fields to `TASK_CASES`; use **Load case** to copy it into editable widgets.

In [ ]:
TASK_CASES = {
    "invoice_review": {
        "label": "Supplier invoice review",
        "task_name": "Review and approve a supplier invoice",
        "task_description": (
            "An accounts payable specialist at a mid-sized manufacturing company "
            "receives an invoice from a supplier for raw materials delivered to the factory. "
            "The invoice includes the supplier's details, purchase order reference, "
            "item descriptions, quantities, unit prices, taxes, and payment terms. "
            "The specialist must determine whether the invoice is legitimate, accurate, "
            "and consistent with the company's purchasing and receiving records. "
            "Discrepancies such as incorrect pricing, missing purchase orders, "
            "quantity mismatches, duplicate invoices, or unrecognized charges may "
            "require clarification from purchasing, warehouse staff, or the supplier. "
            "Once the invoice is sufficiently verified and any exceptions are resolved, "
            "it can be approved for payment according to company policy."
        ),
        "domain": "accounts_payable",
    },

    "customer_support": {
        "label": "SaaS customer support",
        "task_name": "Triage a software customer support ticket",
        "task_description": (
            "A customer support representative at a business-to-business SaaS company "
            "receives a ticket from a customer who reports that several employees "
            "cannot access a reporting dashboard after a recent software update. "
            "The customer provides a brief description of the problem, screenshots, "
            "and the approximate time the issue began. The representative has access "
            "to the customer's subscription details, account history, previous "
            "support tickets, product documentation, and known incident reports. "
            "The representative needs to understand the reported problem, determine "
            "its likely severity and business impact, identify whether a known "
            "solution exists, and decide whether to respond directly or escalate "
            "the issue to technical support or engineering. The customer should "
            "receive an appropriate response explaining the next steps."
        ),
        "domain": "customer_support",
    },

    "production_scheduling": {
        "label": "Manufacturing production scheduling",
        "task_name": "Prepare a weekly production schedule",
        "task_description": (
            "A production planner at a factory that manufactures packaged food "
            "products must prepare next week's production schedule for several "
            "packaging lines. The factory has confirmed customer orders, demand "
            "forecasts, raw material inventories, equipment availability, staffing "
            "schedules, and planned maintenance windows. Different products require "
            "different processing times and packaging configurations, and switching "
            "between products can require equipment cleaning and changeover time. "
            "The planner must produce a feasible schedule that meets delivery "
            "commitments while respecting available capacity, material constraints, "
            "labor availability, and operational priorities. When demand exceeds "
            "capacity or materials are unavailable, the planner must identify "
            "trade-offs and coordinate with production management before releasing "
            "the schedule to the factory floor."
        ),
        "domain": "operations",
    },

    "claims_triage": {
        "label": "Auto insurance claims triage",
        "task_name": "Triage a personal auto collision claim",
        "task_description": (
            "An auto insurance claims intake specialist receives a new claim from "
            "a policyholder who reports a two-vehicle collision at an intersection. "
            "The submitted materials include the policyholder's accident description, "
            "vehicle information, photographs of the damage, the other driver's "
            "contact and insurance details, and a police report number. Some "
            "information may be missing or inconsistent, and a repair estimate "
            "may not yet be available. The specialist must review the claim against "
            "the policy and available records, assess whether the information is "
            "sufficient for further handling, identify missing documents or "
            "potential coverage concerns, and determine the appropriate claims "
            "handling queue. Straightforward claims may follow a standard review "
            "process, while cases involving injuries, disputed circumstances, "
            "potential fraud indicators, or complex coverage questions require "
            "specialized attention. The specialist prepares the claim for the "
            "assigned adjuster rather than making the final coverage or liability decision."
        ),
        "domain": "insurance",
    },
}

assert TASK_CASES


## 3. Shared request-building and execution functions

These functions accept explicit values, do not read widgets, and do not display or mutate notebook state.

In [ ]:
def build_task_request(inputs):
    return TaskDecompositionRequest(task=TaskReference(task_name=inputs["task_name"], task_description=inputs["task_description"]), task_context={"domain": inputs["domain"]})

def build_accounting_input(_inputs, _mode="normalized"):
    raise RuntimeError("The canonical transformation API currently requires explicit normalized retained/gross-removed accounting input. This notebook intentionally does not collect ratios or absolute effort; run task decomposition to inspect LLM-generated relative effort weights.")

def execute_task(inputs, provider):
    return decompose_task(build_task_request(inputs), provider)

def execute_transformation(task_result, inputs, provider, mode="normalized"):
    request = TransformationDecompositionRequest(task_decomposition=task_result, transformation_context={}, accounting_input=build_accounting_input(inputs, mode), support_work_mode=SupportWorkMode.AUTO)
    return decompose_transformation(request, provider)

def task_table(result):
    return pd.DataFrame([{"sequence": s.sequence_index, "subtask_id": s.subtask_id, "subtask": s.subtask_name, "description": s.description, "relative_effort_weight": s.relative_effort_weight} for s in result.operational_decomposition.operational_subtasks])

def allocation_table(result):
    return pd.DataFrame([{"subtask_id": a.subtask_id, "effort": a.effort.value if a.effort else None, "weight_ratio": a.weight_ratio, "source": a.allocation_source} for a in result.baseline_effort_allocations])

def transformation_table(result):
    return pd.DataFrame([{"subtask_id": r.subtask_id, "classification": r.classification.value, "rationale": r.rationale, "baseline_ratio": r.baseline_effort_ratio} for r in result.retain_remove_classification.classified_subtasks])

def snapshot(stage, case_key, provider, inputs, result):
    stamp = datetime.now(timezone.utc).isoformat()
    return {"run_id": _fingerprint((stamp, stage, provider, inputs))[:12], "timestamp": stamp, "case_key": case_key or "edited", "provider": provider, "stage": stage, "inputs": deepcopy(inputs), "result": result.model_dump(mode="json") if hasattr(result, "model_dump") else result}


## 4. Interactive controls and stage-by-stage execution

Select one of the predefined cases and run the stages explicitly.

In [ ]:
# Retire this UI before rebuilding it when this cell is rerun.
for old_button, old_callback in globals().get("_ui_click_bindings", []):
    old_button.on_click(old_callback, remove=True)
for old_widget, old_callback in globals().get("_ui_observer_bindings", []):
    old_widget.unobserve(old_callback, names="value")
for old_widget in reversed(globals().get("_ui_widgets", [])):
    old_widget.close()
clear_output(wait=True)
_ui_click_bindings = []
_ui_observer_bindings = []

provider_widget = widgets.Dropdown(options=list(PROVIDER_ENV), value="OpenAI", description="Provider:", style={"description_width": "180px"})
allow_live_widget = widgets.Checkbox(value=False, description="Allow paid/networked LLM calls")
case_widget = widgets.Dropdown(options=[(v["label"], k) for k, v in TASK_CASES.items()], description="Example case:", style={"description_width": "180px"})
task_name_widget = widgets.HTML()
task_review_widget = widgets.Textarea(description="Human review notes:", layout=widgets.Layout(width="100%", height="60px"), style={"description_width": "180px"})
task_output, transform_output, end_output, history_output = (
    widgets.HTML(layout=widgets.Layout(width="100%", overflow="auto"))
    for _ in range(4)
)
run_task_button = widgets.Button(description="Run task decomposition", button_style="primary")
run_transform_button = widgets.Button(description="Run transformation decomposition", button_style="primary")
run_end_button = widgets.Button(description="Run decompose()", button_style="warning")
run_history_button = widgets.Button(description="Compare saved runs")

def current_task_inputs():
    case = TASK_CASES[case_widget.value]
    return {key: case[key] for key in ("task_name", "task_description", "domain")}
def current_inputs():
    return current_task_inputs()
def invalidate_task(*_):
    state.update(task_result=None, task_fingerprint=None, transformation_result=None, transformation_fingerprint=None)
    task_output.value = ""
    transform_output.value = ""
    end_output.value = ""
def invalidate_transformation(*_):
    state.update(transformation_result=None, transformation_fingerprint=None)
def run_task(_=None):
    # This is an in-flight lock, not a cache: intentional repeated runs are allowed.
    if state.get("task_running"):
        return
    state["task_running"] = True
    run_task_button.disabled = True
    inputs = current_task_inputs()
    provider_label = provider_widget.value
    case_key = case_widget.value
    try:
        result = execute_task(inputs, _provider(provider_label, allow_live_widget.value))
        state.update(task_result=result, task_fingerprint=_fingerprint(inputs), transformation_result=None, transformation_fingerprint=None)
        state["runs"].append(snapshot("task", case_key, provider_label, inputs, result))
        _set_panel(task_output, task_table(result))
        transform_output.value = ""
        end_output.value = ""
    except Exception as exc:
        state.update(task_result=None, task_fingerprint=None, transformation_result=None, transformation_fingerprint=None)
        _error(task_output, exc)
    finally:
        state["task_running"] = False
        run_task_button.disabled = False
def run_transform(_=None):
    inputs = current_inputs()
    if state["task_result"] is None or state["task_fingerprint"] != _fingerprint(current_task_inputs()): return _error(transform_output, RuntimeError("Run task decomposition again: the task result is missing or stale."))
    try:
        build_accounting_input(inputs)
        result = execute_transformation(state["task_result"], inputs, _provider(provider_widget.value, allow_live_widget.value))
        state.update(transformation_result=result, transformation_fingerprint=_fingerprint(inputs))
        state["runs"].append(snapshot("transformation", case_widget.value, provider_widget.value, inputs, result))
        _set_panel(transform_output, transformation_table(result))
    except Exception as exc: _error(transform_output, exc)
def run_end_to_end(_=None):
    inputs = current_inputs()
    try:
        build_accounting_input(inputs)
        provider = _provider(provider_widget.value, allow_live_widget.value)
        result = decompose(build_task_request(inputs), provider, transformation_context={}, accounting_input=build_accounting_input(inputs))
        state["runs"].append(snapshot("end_to_end", case_widget.value, provider_widget.value, inputs, result))
        _set_panel(end_output, pd.DataFrame([{"provider": provider_widget.value, "subtasks": len(result.task_decomposition.operational_decomposition.operational_subtasks), "W0": result.accounting.w0, "W1": result.accounting.w1, "effect": result.accounting.effect.value}]))
    except Exception as exc: _error(end_output, exc)
def show_history(_=None):
    rows = []
    for run in state["runs"]:
        result = run["result"]
        task = result.get("task_decomposition", result) if isinstance(result, dict) else {}
        subtasks = task.get("operational_decomposition", {}).get("operational_subtasks")
        rows.append({"run_id": run["run_id"], "case": run["case_key"],
                     "provider": run["provider"], "stage": run["stage"],
                     "subtasks": len(subtasks) if subtasks is not None else None})
    _set_panel(history_output, pd.DataFrame(rows))

def update_task_name(_=None):
    case = TASK_CASES[case_widget.value]
    task_name_widget.value = (
        f"<h3>{escape(case['task_name'])}</h3>"
        f"<p>{escape(case['task_description'])}</p>"
    )

_ui_observer_bindings = [
    (case_widget, invalidate_task),
    (provider_widget, invalidate_task),
    (case_widget, update_task_name),
]
for widget, callback in _ui_observer_bindings:
    widget.observe(callback, names="value")
_ui_click_bindings = [
    (run_task_button, run_task),
    (run_transform_button, run_transform),
    (run_end_button, run_end_to_end),
    (run_history_button, show_history),
]
for button, callback in _ui_click_bindings:
    button.on_click(callback)
update_task_name()
# Review notes are mounted once, outside any result panel.
ui = widgets.VBox([
    provider_widget, allow_live_widget, case_widget, task_name_widget,
    run_task_button, task_output, widgets.HTML("<h3>Human review</h3>"), task_review_widget,
    run_transform_button, transform_output, run_end_button, end_output,
    run_history_button, history_output,
])
_ui_widgets = [ui, *ui.children]
invalidate_task()
display(ui)


## 5. Added-work estimation and support-work resolution

The transformation result contains provider classifications and the canonical support-work resolution. This notebook displays those results; it does not reimplement support-work formulas.

## 6. Deterministic accounting

The canonical accounting API currently requires an explicit normalized or absolute accounting input. This notebook intentionally supplies neither: it is focused on testing provider-generated relative effort weights without user-entered effort or ratios.

In [ ]:
display(Markdown("Transformation/accounting execution is intentionally blocked here until the canonical API can derive normalized accounting input from provider outputs. No absolute effort or manually entered ratio is used."))


## 7. End-to-end `decompose()` execution

This explicit button composes the canonical task and transformation stages. It uses the current editable inputs and does not reuse a stale transformation result.

## 8. Provider comparison and repeatability

Run one provider at a time with the explicit task button, then inspect the preserved `state['runs']` history. Counts are operational summaries only and are not semantic quality scores.